# ChillNet — Analyse exploratoire du jeu d'entraînement « vigilance canicule »

Ce notebook documente la **distribution et la cohérence** du jeu de données qui
alimente `scripts/train.py`. Il utilise `app.dataset.build_dataset()` — la même
source de vérité que l'entraînement — donc toute évolution des features ou de
l'étiquetage s'y reflète automatiquement.

> Dépendances d'exploration **optionnelles** (non requises en production) :
> `pip install pandas matplotlib seaborn`.

In [ ]:
import sys
from pathlib import Path

# Rendre `app.*` importable quand le notebook est lancé depuis ai-service/notebooks.
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))

import numpy as np
from app import dataset
from app.features import FEATURE_ORDER

ds = dataset.build_dataset(with_noise=False)
X, y = ds["X"], ds["y"]
print("lignes :", X.shape[0], "| features :", X.shape[1])
print("hash des données :", ds["hash"])
print("features :", FEATURE_ORDER)

## 1. Distribution des classes

L'étiquetage combine règles documentées (`config/labeler.yaml`) et bruit
gaussien déterministe. On attend une majorité de `normal`/`jaune` sur un
historique estival tempéré, avec `orange`/`rouge` plus rares mais présents.

In [ ]:
import pandas as pd

valeurs, comptes = np.unique(y, return_counts=True)
distribution = pd.Series(comptes, index=valeurs).reindex(
    dataset.CLASSES, fill_value=0
)
distribution.to_frame("n")
distribution.plot(kind="bar", title="Distribution des classes", color="#F59E0B")

# Part de chaque classe (contrôle du déséquilibre).
(distribution / distribution.sum() * 100).round(1).to_frame("%")

## 2. Statistiques descriptives des features

Vérifie l'absence de valeurs aberrantes et l'échelle de chaque variable.
`max_24h`/`max_48h`/`heure_pic` sont approchées par la température du jour à
l'entraînement (historique journalier, cf. README).

In [ ]:
df = pd.DataFrame(X, columns=FEATURE_ORDER)
df["niveau"] = y
df.describe().round(2)

## 3. Température vs niveau

Répartition de la température par niveau : confirme la monotonie attendue
(les niveaux graves concentrent les températures hautes).

In [ ]:
df.boxplot(column="temp", by="niveau", grid=False, figsize=(7, 4))
import matplotlib.pyplot as plt

plt.suptitle("")
plt.title("Température par niveau")
plt.ylabel("°C")
plt.show()

df.groupby("niveau")["temp"].agg(["count", "min", "mean", "max"]).round(2)

## 4. Corrélations entre features

Repère les features redondantes (ex. `temp` / `heat_index` / `max_24h` sont
naturellement corrélées sur un historique journalier).

In [ ]:
correlations = df[FEATURE_ORDER].corr().round(2)
correlations

## 5. Synthèse

- Le jeu est **déséquilibré** (canicule rare) → `class_weight="balanced"` côté modèle.
- L'étiquetage est **déterministe** à graine fixe : le notebook ne modifie pas les données.
- La **monotonie** température → niveau doit tenir : garde-fou du test `tests/test_labeler.py`.

Prochaine étape : `python scripts/train.py` puis `python scripts/evaluate.py`.